# Kuku Promo Finder (Problem 1) — Colab demo

Finds 20–60s promo clips in timestamped microdrama screenplays: hook in the
first 3s, ends on a cliffhanger, makes sense standalone. **The input pack is
Kuku's own content — upload your zip below; nothing is stored or published.**

Model swap = edit one cell (`MODEL_CONFIG`). With `backend="openai"` the calls
go straight to any OpenAI-compatible endpoint (Gemini, Groq, OpenAI, Kuku APIs).
With `backend="file"` the prompts are exported and responses pasted back.

In [ ]:
#@title 1. Upload the input pack zip (Kuku's content, stays in this session)
from google.colab import files
up = files.upload()  # select the input-pack zip
import zipfile, pathlib
zpath = next(iter(up))
zipfile.ZipFile(zpath).extractall('input')
txts = list(pathlib.Path('input').rglob('episode-*.txt'))
print(f'{len(txts)} episode files extracted')

In [ ]:
#@title 2. Clone pipeline + configure model (swap = edit this cell)
!git clone https://github.com/<your-user>/kuku-promo-finder.git 2>/dev/null || echo 'repo already cloned'
%cd kuku-promo-finder
MODEL_CONFIG = {  # <-- one-line model swap
    "backend": "openai",            # or "file"
    "name": "gemini-2.0-flash",
    "base_url": "https://generativelanguage.googleapis.com/v1beta/openai",
    "api_key_env": "GEMINI_API_KEY",
}
import json, pathlib
cfg = json.loads(pathlib.Path('config.json').read_text())
cfg['candidate_model'] = {**cfg['candidate_model'], **MODEL_CONFIG}
cfg['judge_model'] = {**cfg['judge_model'], **MODEL_CONFIG}
pathlib.Path('config.json').write_text(json.dumps(cfg, indent=2))
print('config:', MODEL_CONFIG)

In [ ]:
#@title 3. Run the pipeline end-to-end
!python3 -I src/run.py prepare   # parse + candidate prompts
# With backend=openai the responses are fetched directly; with backend=file,
# export prompts/ and paste responses into prompts/raw/*.response.json.
!python3 -I src/merge_responses.py
!python3 -I src/run.py mid       # validate + season combos
!python3 -I src/run.py judge
!python3 -I src/run.py rank
from google.colab import files
files.download('outputs/promo_report.md')
files.download('outputs/ranked_promos.csv')

Outputs: `outputs/promo_report.md` (ranked promos with timestamps, hook,
cliffhanger, why-it-works), `ranked_promos.csv`, `quality_report.md`, and
`prompts/` = the unedited AI log.